---

# 6 · Summary of a Full-Length Recording
### Naif Aldosari

Sections 2–4 work on short samples: a single utterance, a few turns, a
two-minute excerpt. **This section takes one complete recording — half an hour
to an hour of real speech — and returns a brief you can put on a slide.**

That is a different problem, not a longer one:

| At two minutes | At thirty minutes |
|---|---|
| ~300 words — one model call | ~5,000 words — past every summariser's input limit |
| Every sentence can be shown | The brief has to *choose*, and justify the choice |
| One topic | Several, in sequence, and the order matters |

**How it is solved here.** The transcript is cut into windows that fit the
model, each window is summarised, and the window summaries are summarised
again — repeatedly, until the whole recording fits in one pass. On top of that
abstractive pass sits an extractive one that scores every real sentence in the
transcript and keeps the highest-information ones, **each with the timestamp it
was spoken at**. So the headline is written by the model, and the points under
it are the meeting's own words with a time you can jump to and check.

**Two outputs, for two slides:**

1. `vocalyze_summary.txt` / the rendered card below — the brief, sized for a slide.
2. `vocalyze_recording.mp3` — the recording itself, for the slide before it.

Run the cells in order. On a T4 GPU the whole section takes about six minutes
for a thirty-minute recording.

### 6.1 · Setup

Everything the section needs, and the three settings worth changing.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 · Setup
# ---------------------------------------------------------------------------
!pip install -q openai-whisper transformers 2>/dev/null

import json, math, os, re, subprocess, textwrap, time, urllib.request
from collections import Counter
from pathlib import Path

import torch

WORK = Path("/content/vocalyze_summary")
WORK.mkdir(parents=True, exist_ok=True)

# --- settings ---------------------------------------------------------------

# How much of the recording to process. None means all of it. Set this to 10
# while rehearsing so a run takes one minute instead of six; set it back to
# None for the real thing.
MINUTES_TO_PROCESS = None

# Whisper size. "small" is the accuracy/latency point that fits a live demo on
# a T4; "medium" is better and roughly three times slower.
WHISPER_MODEL = "small"

# The abstractive summariser. bart-large-cnn is trained on long documents and
# takes 1024 tokens at a time, which is what makes the map-reduce below cheap.
# If it cannot be downloaded the code falls back to the flan-t5-base already
# used in section 4.
SUMMARY_MODEL = "facebook/bart-large-cnn"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"device      : {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
print(f"whisper     : {WHISPER_MODEL}")
print(f"summariser  : {SUMMARY_MODEL}")
print(f"processing  : {'the whole recording' if MINUTES_TO_PROCESS is None else str(MINUTES_TO_PROCESS) + ' minutes'}")
if DEVICE == "cpu":
    print("\n!! CPU runtime — a 30-minute recording will take ~40 minutes to transcribe.")
    print("   Runtime -> Change runtime type -> T4 GPU, then run this cell again.")

### 6.2 · The recording

A half-hour of real, unscripted, multi-speaker English: one meeting from the
**AMI Meeting Corpus** — the same corpus section 1 builds its splits from, so
the recording summarised here is drawn from the project's own dataset rather
than from somewhere unrelated.

The mirror is tried meeting by meeting until one downloads, and the duration
printed below is measured from the file itself, not assumed. If the mirror is
unreachable on the day, set `AUDIO_URL` to any direct link, or drop a file into
`/content/` and set `AUDIO_PATH` — the rest of the section does not care where
the audio came from.

In [ ]:
# ---------------------------------------------------------------------------
# 6.2 · Fetch the recording, normalise it, and make the file for the slide
# ---------------------------------------------------------------------------

# AMI scenario meetings, longest first. Each is four people, one room, real
# speech with interruptions and crosstalk — a genuine test, unlike read audio.
AMI_MIRROR = "https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus"
CANDIDATES = ["ES2004c", "ES2004b", "ES2004d", "IS1000a", "ES2002a"]

AUDIO_URL  = None   # set this to override the mirror with any direct link
AUDIO_PATH = None   # set this to use a file already on the runtime

source_wav = WORK / "source.wav"


def _duration(path):
    """Seconds of audio in a file, read from the file rather than assumed."""
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=nw=1:nk=1", str(path)],
        capture_output=True, text=True,
    )
    try:
        return float(out.stdout.strip())
    except ValueError:
        return 0.0


def _hms(seconds):
    seconds = int(round(seconds))
    return f"{seconds // 3600:d}:{seconds % 3600 // 60:02d}:{seconds % 60:02d}"


# --- 1. find an audio file --------------------------------------------------
if AUDIO_PATH:
    origin, downloaded = Path(AUDIO_PATH), Path(AUDIO_PATH)
    print(f"using the local file {downloaded}")
else:
    urls = [AUDIO_URL] if AUDIO_URL else [
        f"{AMI_MIRROR}/{m}/audio/{m}.Mix-Headset.wav" for m in CANDIDATES
    ]
    downloaded = None
    for url in urls:
        target = WORK / url.rsplit("/", 1)[-1]
        try:
            print(f"trying {url.rsplit('/', 1)[-1]} ... ", end="", flush=True)
            urllib.request.urlretrieve(url, target)
            size_mb = target.stat().st_size / 1e6
            if size_mb < 1:
                print("too small, skipping")
                continue
            print(f"{size_mb:.0f} MB")
            downloaded, origin = target, url
            break
        except Exception as exc:
            print(f"unavailable ({type(exc).__name__})")
    if downloaded is None:
        raise SystemExit(
            "No recording could be downloaded.\n"
            "Set AUDIO_URL to a direct link, or upload a file to /content/ and "
            "set AUDIO_PATH to it, then run this cell again."
        )

full_seconds = _duration(downloaded)

# --- 2. normalise to what Whisper wants, trimming if asked ------------------
trim = ["-t", str(int(MINUTES_TO_PROCESS * 60))] if MINUTES_TO_PROCESS else []
subprocess.run(
    ["ffmpeg", "-y", "-v", "error", "-i", str(downloaded), *trim,
     "-ac", "1", "-ar", "16000", str(source_wav)],
    check=True,
)
used_seconds = _duration(source_wav)

# --- 3. the file that goes on the slide -------------------------------------
# 64 kbps mono is transparent for speech and turns half an hour into ~14 MB,
# which a slide deck can carry. The full-size WAV cannot.
slide_mp3 = WORK / "vocalyze_recording.mp3"
subprocess.run(
    ["ffmpeg", "-y", "-v", "error", "-i", str(source_wav),
     "-codec:a", "libmp3lame", "-b:a", "64k", str(slide_mp3)],
    check=True,
)

# A one-minute excerpt, so the notebook can play something without embedding
# fourteen megabytes of base64 in the shared file.
preview_mp3 = WORK / "preview.mp3"
subprocess.run(
    ["ffmpeg", "-y", "-v", "error", "-i", str(source_wav), "-t", "60",
     "-codec:a", "libmp3lame", "-b:a", "64k", str(preview_mp3)],
    check=True,
)

print()
print(f"source        {str(origin).rsplit('/', 1)[-1]}")
print(f"full length   {_hms(full_seconds)}")
print(f"processing    {_hms(used_seconds)}  ({used_seconds / 60:.1f} minutes)")
print(f"normalised    16 kHz mono WAV, {source_wav.stat().st_size / 1e6:.0f} MB")
print(f"for the slide {slide_mp3.name}, {slide_mp3.stat().st_size / 1e6:.1f} MB")

from IPython.display import Audio, display
print("\nfirst minute:")
display(Audio(str(preview_mp3)))

### 6.3 · Transcription

Whisper over the whole recording in one call — it does its own windowing and
carries context across window boundaries, which is why the transcript is not
cut into pieces before this point. The result is cached to disk, so a second
run of the cells below costs nothing.

In [ ]:
# ---------------------------------------------------------------------------
# 6.3 · Transcribe
# ---------------------------------------------------------------------------
import whisper

cache = WORK / f"transcript-{WHISPER_MODEL}-{int(used_seconds)}.json"

if cache.exists():
    transcript = json.loads(cache.read_text())
    print(f"loaded the cached transcript ({cache.name})")
else:
    print(f"loading whisper-{WHISPER_MODEL} ...")
    asr = whisper.load_model(WHISPER_MODEL, device=DEVICE)

    print(f"transcribing {used_seconds / 60:.1f} minutes — this is the slow step")
    started = time.time()
    raw = asr.transcribe(
        str(source_wav),
        language="en",
        fp16=(DEVICE == "cuda"),
        condition_on_previous_text=False,   # stops one bad window derailing the rest
        verbose=False,
    )
    elapsed = time.time() - started

    transcript = {
        "elapsed": elapsed,
        "segments": [
            # Whisper emits a typographic apostrophe about as often as an ASCII
            # one; normalising here means nothing downstream has to match both.
            {"start": float(s["start"]), "end": float(s["end"]),
             "text": s["text"].strip().replace("’", "'")}
            for s in raw["segments"] if s["text"].strip()
        ],
    }
    cache.write_text(json.dumps(transcript))
    print(f"done in {elapsed / 60:.1f} minutes "
          f"({used_seconds / elapsed:.1f}x real time)")

segments = transcript["segments"]
words = sum(len(s["text"].split()) for s in segments)

print()
print(f"segments   {len(segments):,}")
print(f"words      {words:,}")
print(f"speech     {sum(s['end'] - s['start'] for s in segments) / 60:.1f} minutes "
      f"of {used_seconds / 60:.1f}")
print("\nopening lines:")
for s in segments[:5]:
    print(f"  [{_hms(s['start'])}]  {s['text'][:96]}")

### 6.4 · The summary

Three passes over the transcript, each answering a different question.

| Pass | Question | Method |
|---|---|---|
| **Abstractive** | What was this about? | windows of ~700 words → model → summarise the summaries, until one window is left |
| **Cue-phrase** | What was committed to? | the phrases people actually use to commit — *we'll*, *let's*, *I'll*, *next step*, *by Friday* |
| **Extractive** | Which other moments mattered? | every line scored by how much rare, meeting-specific vocabulary it carries |

The abstractive pass writes the headline; the other two fill the body with the
recording's own words, each carrying the time it was said. Nothing on the slide
is unsourced.

Two rules keep the body honest, and both were added because the first version
broke them. **Nothing appears twice** — the cue-phrase pass runs first and the
extractive pass is told what it already took. **Nothing clusters** — the
recording is divided into as many parts as there are bullets, and each part has
to be represented before any part gets a second one. Without that, a
seventy-minute meeting returns seven points from its noisiest five minutes,
which reads as a confident summary of the wrong thing.

In [ ]:
# ---------------------------------------------------------------------------
# 6.4 · Summarise
# ---------------------------------------------------------------------------
from transformers import pipeline as hf_pipeline

# Imported under its own name on purpose: section 3 binds `pipeline` to the
# pyannote object, and the bare name here would call the diarizer.
try:
    summariser = hf_pipeline("summarization", model=SUMMARY_MODEL,
                             device=0 if DEVICE == "cuda" else -1)
    ABSTRACTIVE_WINDOW = 700
except Exception as exc:
    print(f"{SUMMARY_MODEL} unavailable ({exc}); falling back to flan-t5-base")
    summariser = hf_pipeline("text2text-generation", model="google/flan-t5-base",
                             device=0 if DEVICE == "cuda" else -1)
    ABSTRACTIVE_WINDOW = 300

FULL_TEXT = " ".join(s["text"] for s in segments)


# --- pass 1 · abstractive, by map-reduce ------------------------------------
def _windows(text, size):
    """Split on word count, at sentence ends where one is near enough."""
    parts, current = [], []
    for sentence in re.split(r"(?<=[.!?])\s+", text):
        current.append(sentence)
        if sum(len(p.split()) for p in current) >= size:
            parts.append(" ".join(current))
            current = []
    if current:
        parts.append(" ".join(current))
    return parts


def _summarise(text, target):
    if hasattr(summariser, "task") and summariser.task == "summarization":
        out = summariser(text, max_length=target, min_length=target // 3,
                         do_sample=False, truncation=True)
        return out[0]["summary_text"].strip()
    out = summariser(f"Summarise this meeting excerpt:\n{text}",
                     max_length=target, truncation=True)
    return out[0]["generated_text"].strip()


def abstractive(text):
    """Summarise, then summarise the summaries, until one window is left."""
    level, passes = _windows(text, ABSTRACTIVE_WINDOW), 0
    while True:
        passes += 1
        print(f"  pass {passes}: {len(level)} window(s), "
              f"{sum(len(w.split()) for w in level):,} words")
        summaries = [_summarise(w, 110) for w in level]
        joined = " ".join(summaries)
        if len(summaries) == 1:
            return joined
        level = _windows(joined, ABSTRACTIVE_WINDOW)
        if len(level) == 1 and passes > 1:
            return _summarise(level[0], 150)


print("abstractive pass")
started = time.time()
headline = abstractive(FULL_TEXT)
print(f"  {time.time() - started:.0f}s\n")


# --- vocabulary shared by passes 2 and 3 ------------------------------------
STOP = set("""a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few
for from further had has have having he her here hers him his how i if in into is it its just
me more most my no nor not now of off on once only or other our out over own same she should so
some such than that the their them then there these they this those through to too under until
up very was we were what when where which while who whom why will with would you your yeah yes
okay ok right um uh hmm mm like know think going get got just really actually thing things
""".split())
_WORD = re.compile(r"[a-z][a-z'\-]+")


def _content(text):
    return [w for w in _WORD.findall(text.lower()) if w not in STOP and len(w) > 2]


def _idf(segments):
    """How rare each word is across the recording. A word in every segment
    says nothing about any one of them."""
    document_frequency = Counter()
    for s in segments:
        document_frequency.update(set(_content(s["text"])))
    n = max(len(segments), 1)
    return {w: math.log(n / (1 + df)) for w, df in document_frequency.items()}


IDF = _idf(segments)


def _pick(pool, k, span, taken_ids=(), avoid=()):
    """The k strongest lines in `pool`: no near-duplicates, and no more than
    one from each k-th of the recording until every part has had a turn.

    Without the spread a long meeting returns seven points from its noisiest
    five minutes, which reads as a summary of the wrong thing.
    """
    scored = []
    for s in pool:
        if id(s) in taken_ids:
            continue
        terms = _content(s["text"])
        if len(terms) < 5:                      # "yeah exactly" is not a moment
            continue
        unique = set(terms)
        scored.append((sum(IDF.get(w, 0) for w in unique) / math.sqrt(len(terms)),
                       s, unique))
    scored.sort(key=lambda row: -row[0])

    bucket_width = max(span / k, 1e-6)
    chosen, seen, buckets = [], list(avoid), set()
    for one_per_bucket in (True, False):
        for _score, s, unique in scored:
            if len(chosen) == k:
                break
            if any(s is c for c in chosen):
                continue
            bucket = min(int(s["start"] // bucket_width), k - 1)
            if one_per_bucket and bucket in buckets:
                continue
            if any(len(unique & other) / max(len(unique | other), 1) > 0.45
                   for other in seen):
                continue
            chosen.append(s)
            seen.append(unique)
            buckets.add(bucket)
    return sorted(chosen, key=lambda s: s["start"])


# --- pass 2 · what was committed to -----------------------------------------
# Run before the key moments so a commitment is never printed twice: it belongs
# under "Commitments", and the extractive pass is told to look elsewhere.
# The contraction is a separate alternative on purpose. Written as
# `we (?:will|'ll)` the space is required, so "we'll" — much the commonest way
# anyone commits to anything out loud — never matches and the block comes back
# nearly empty on a real recording.
COMMITMENT = re.compile(
    r"\b(?:we|i|you)'ll\b"
    r"|\b(?:we|i|you) (?:will|should|need to|have to|are going to|agreed|decided)\b"
    r"|\blet's\b"
    r"|\bthe (?:decision|plan) is\b"
    r"|\baction item\b|\bnext step\b"
    r"|\bby (?:monday|tuesday|wednesday|thursday|friday|next week|the end of)\b",
    re.I,
)

actions = _pick([s for s in segments if COMMITMENT.search(s["text"])], 5, used_seconds)

# --- pass 3 · the moments that carried the most information ------------------
# Commitments are excluded outright rather than only de-duplicated: they have
# their own block below, and a line that reads like one belongs there whether
# or not it was the line chosen.
elsewhere = [s for s in segments if not COMMITMENT.search(s["text"])]
moments = _pick(elsewhere if len(elsewhere) > 40 else segments, 7, used_seconds,
                taken_ids={id(s) for s in actions},
                avoid=[set(_content(s["text"])) for s in actions])

print(f"headline    {len(headline.split())} words")
print(f"key moments {len(moments)}")
print(f"commitments {len(actions)}")

### 6.5 · The slide

The card below is sized and spaced to be screenshotted straight into the deck.
The same brief is printed as plain text underneath, for pasting into a text
placeholder instead.

In [ ]:
# ---------------------------------------------------------------------------
# 6.5 · Render the brief
# ---------------------------------------------------------------------------
from IPython.display import HTML, display


def _sentence_case(text):
    text = re.sub(r"\s+", " ", text).strip().rstrip(",;:")
    if text and not text.endswith((".", "!", "?")):
        text += "."
    return text[:1].upper() + text[1:]


title = f"Meeting brief — {_hms(used_seconds)} recording"
stats = [
    ("Recording", _hms(used_seconds)),
    ("Words transcribed", f"{words:,}"),
    ("Compressed to", f"{len(headline.split()) + sum(len(m['text'].split()) for m in moments):,} words"),
    ("Ratio", f"{words / max(len(headline.split()) + sum(len(m['text'].split()) for m in moments), 1):.0f}:1"),
]

rows = "".join(
    f"<div style='display:flex;gap:14px;padding:9px 0;border-top:1px solid #E8E6E1'>"
    f"<span style='font-variant-numeric:tabular-nums;color:#9A958C;font-size:13px;"
    f"min-width:62px;padding-top:2px'>{_hms(m['start'])}</span>"
    f"<span style='flex:1'>{_sentence_case(m['text'])}</span></div>"
    for m in moments
)
action_rows = "".join(
    f"<div style='display:flex;gap:14px;padding:9px 0;border-top:1px solid #E8E6E1'>"
    f"<span style='font-variant-numeric:tabular-nums;color:#9A958C;font-size:13px;"
    f"min-width:62px;padding-top:2px'>{_hms(a['start'])}</span>"
    f"<span style='flex:1'>{_sentence_case(a['text'])}</span></div>"
    for a in actions
) or "<div style='padding:9px 0;border-top:1px solid #E8E6E1;color:#9A958C'>" \
     "Nothing in this recording was phrased as a commitment.</div>"

chips = "".join(
    f"<span style='display:inline-block;margin-right:22px'>"
    f"<b style='font-variant-numeric:tabular-nums'>{value}</b>"
    f"<span style='color:#9A958C'> {label.lower()}</span></span>"
    for label, value in stats
)

display(HTML(f"""
<div style="max-width:820px;background:#FDFCFA;color:#1A1917;border:1px solid #E8E6E1;
            border-radius:14px;padding:34px 38px;font-family:-apple-system,Segoe UI,Inter,sans-serif;
            line-height:1.62;font-size:15.5px">
  <div style="font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:#B4530A">
    Vocalyze &nbsp;·&nbsp; automatic brief
  </div>
  <h2 style="margin:.35em 0 .55em;font-size:25px;font-weight:600;letter-spacing:-.01em">{title}</h2>
  <p style="margin:0 0 26px;font-size:16.5px">{_sentence_case(headline)}</p>

  <div style="font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:#9A958C;
              margin-bottom:4px">Key moments</div>
  {rows}

  <div style="font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:#9A958C;
              margin:26px 0 4px">Commitments</div>
  {action_rows}

  <div style="margin-top:28px;padding-top:16px;border-top:1px solid #E8E6E1;font-size:13.5px">
    {chips}
  </div>
</div>
"""))

# --- the same brief as plain text, for a text placeholder -------------------
lines = [title, "=" * len(title), "", textwrap.fill(_sentence_case(headline), 78), "", "KEY MOMENTS"]
lines += [f"  [{_hms(m['start'])}]  {_sentence_case(m['text'])}" for m in moments]
lines += ["", "COMMITMENTS"]
lines += [f"  [{_hms(a['start'])}]  {_sentence_case(a['text'])}" for a in actions] or ["  (none stated)"]
lines += ["", "  ".join(f"{v} {k.lower()}" for k, v in stats)]

brief_text = "\n".join(lines)
(WORK / "vocalyze_summary.txt").write_text(brief_text)
print(brief_text)

### 6.6 · Files for the deck

Two downloads: the recording for one slide, the brief for the next. Colab
saves both to your machine.

In [ ]:
# ---------------------------------------------------------------------------
# 6.6 · Hand over the files
# ---------------------------------------------------------------------------
deliverables = [slide_mp3, WORK / "vocalyze_summary.txt"]
(WORK / "vocalyze_transcript.txt").write_text(
    "\n".join(f"[{_hms(s['start'])}]  {s['text']}" for s in segments)
)
deliverables.append(WORK / "vocalyze_transcript.txt")

for path in deliverables:
    print(f"{path.stat().st_size / 1e6:>7.2f} MB   {path.name}")

try:
    from google.colab import files
    for path in deliverables:
        files.download(str(path))
except ImportError:
    print(f"\nNot on Colab — the files are in {WORK}")